# 06 — Spatial correspondence and overlap
Figure 2 and Supplementary figures S6, S7, S9, S15, S19, S20.
Cross-domain similarity: Dice of pFDR < 0.05 maps and Pearson r of unthresholded in-brain T;
alignment with the normative connectome degree map (binary: top 5 % of voxels).

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from mapstats import *
from figstyle import apply_figure_style, homogenize_fig
from plotting import plot_combined_multimodal_figure, plot_binary_comparison
import matplotlib.pyplot as plt

apply_figure_style()
brain, affine, BG = load_mask()
bg_img = nib.load(BG)
degree = nib.load(DEGREE_MAP).get_fdata()
degree_bin = degree > np.percentile(degree[np.isfinite(degree)], 95)
NAMES_ = {"degree": "Degree", **DOMAINS}

def save(fig, name):
    homogenize_fig(fig); fig.savefig(FIG_DIR / name, dpi=300, bbox_inches="tight"); plt.close(fig)

In [ ]:
summary = []
for family, pmap, name in [(LNM, F_PERM_FDR, "Fig2_crossdomain_lnm.png"),
                           (LNM, F_PARAM_FDR, "S06_crossdomain_lnm_parametric.png"),
                           (OS, F_PERM_FDR, "S09_crossdomain_onesample.png")]:
    bd = {k: sigmap(family, k, brain, pmap) for k in KEYS}
    cd = {k: tmap(family, k, brain) for k in KEYS}
    fig = plot_combined_multimodal_figure({**bd, "degree": degree_bin}, {**cd, "degree": degree}, affine, bg_img=bg_img,
                                          label_rename_dict=NAMES_, fontsize=14, figsize_base=3,
                                          dot_size_factor=1600, label_wrap_width=20)
    save(fig, name)
    D = matrix(lambda a, b: 1.0 if a == b else dice(bd[a], bd[b]), KEYS)
    R = matrix(lambda a, b: 1.0 if a == b else pearson(cd[a], cd[b]), KEYS)
    for domains, keys in (("all six", KEYS), ("significant", [k for k in KEYS if bd[k].any()])):
        Ds, Rs = offdiag(D.loc[keys, keys]), offdiag(R.loc[keys, keys])
        dd = np.array([dice(bd[k], degree_bin) for k in keys]); rd = np.array([pearson(cd[k], degree) for k in keys])
        summary.append({"figure": name, "domains": domains, "n": len(keys),
                        "dice_mean": Ds.mean(), "dice_sd": Ds.std(), "r_mean": Rs.mean(), "r_sd": Rs.std(),
                        "degree_dice_mean": dd.mean(), "degree_dice_sd": dd.std(),
                        "degree_r_mean": rd.mean(), "degree_r_sd": rd.std()})
    R.to_csv(TAB_DIR / f"crossdomain_pearson__{family}__{pmap.split('_')[1]}.csv")
    D.to_csv(TAB_DIR / f"crossdomain_dice__{family}__{pmap.split('_')[1]}.csv")
pd.DataFrame(summary).round(3).to_csv(TAB_DIR / "crossdomain_summary.csv", index=False)
pd.DataFrame(summary).round(3)

Per-domain overlap of two significance maps.

In [ ]:
overlaps = []
for (fa, pa, na), (fb, pb, nb), name in [
        ((LNM, F_PERM_FDR, "Label permutation-based"), (LNM, F_PARAM_FDR, "Parametric"), "S07_overlap_perm_vs_param.png"),
        ((LNM, F_PERM_FDR, "Label permutation-based"), (OS, F_PERM_FDR, "One-sample t-test"),
         "S15_overlap_casecontrol_vs_onesample.png"),
        ((LNM, F_PERM_FDR, "LNM"), (VLSM, F_PERM_FDR, "VLSM"), "S19_overlap_lnm_vs_vlsm.png"),
        ((LNM_FULL, F_PERM_FDR, "LNM"), (VLSM_FULL, F_PERM_FDR, "VLSM"), "S20_overlap_lnm_vs_vlsm_volnihss.png")]:
    A = {k: sigmap(fa, k, brain, pa) for k in KEYS}
    B = {k: sigmap(fb, k, brain, pb) for k in KEYS}
    fig, _, d, keys = plot_binary_comparison(A, B, affine, label_rename_dict=DOMAINS, name1=na, name2=nb,
                                             fontsize=20, bg_img=bg_img)
    save(fig, name)
    overlaps += [{"figure": name, "domain": DOMAINS[k], "voxels_a": int(A[k].sum()), "voxels_b": int(B[k].sum()),
                  "dice": round(float(x), 3)} for k, x in zip(keys, d)]
overlaps = pd.DataFrame(overlaps)
overlaps.to_csv(TAB_DIR / "overlap_dice.csv", index=False)
# mean Dice over domains in which both maps contain significant voxels
both = overlaps[(overlaps.voxels_a > 0) & (overlaps.voxels_b > 0)]
both.groupby("figure").dice.agg(mean="mean", sd=lambda x: x.std(ddof=0), n="count").round(2)

LNM adjusted for co-occurring impairments (S23): correlation with the unadjusted map of the same domain and
among the three domains before and after adjustment (unthresholded in-brain T).

In [ ]:
T0 = {k: tmap(LNM, k, brain) for k in XDOM_KEYS}
T1 = {k: tmap(LNM_XDOM, k, brain) for k in XDOM_KEYS}
print("adjusted vs unadjusted:", {SH[k]: round(pearson(T1[k], T0[k]), 2) for k in XDOM_KEYS})
for a, b in [(0, 1), (0, 2), (1, 2)]:
    ka, kb = XDOM_KEYS[a], XDOM_KEYS[b]
    print(f"{SH[ka]} ~ {SH[kb]}: r = {pearson(T0[ka], T0[kb]):.2f} -> {pearson(T1[ka], T1[kb]):.2f}")